# @examples/pvlib-python

A set of documented functions for simulating the performance of photovoltaic energy systems.

Published version **v1.0.0** — this notebook fetches that exact snapshot from the
PowerAI Hub, not the upstream repository's latest commit.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9).

The Hub does not run primitives. This notebook runs in *your* Colab session, on
Google's hardware, under your account.

[View on the Hub](https://hub.powerai.ai/examples/pvlib-python)


In [ ]:
PRIMITIVE = "@examples/pvlib-python"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/examples/pvlib-python/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import subprocess, sys
from pathlib import Path

req = next(Path(WORKDIR).rglob("requirements.txt"), None)
if req is None:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
else:
    print(f"Installing {req.relative_to(WORKDIR)} …")
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        capture_output=True, text=True,
    )
    print("done" if done.returncode == 0 else f"pip failed:\n{done.stderr[-2000:]}")


In [ ]:
import os
os.chdir(WORKDIR)

readme = next((p for p in sorted(Path(WORKDIR).rglob("README*")) if p.is_file()), None)
print(readme.read_text(errors="replace")[:3000] if readme else "No README in this primitive.")


## Serve this primitive as an endpoint

The cells below install the archive you just unpacked, start the endpoint that answers
this primitive's declared tool interface, and open a public HTTPS tunnel to it — so the
Hub, an agent, or a `curl` can call it and get a real computed answer.

Two things to know before you rely on it:

* **It dies with this session.** Colab reclaims idle notebooks after ~90 minutes and
  caps sessions around 12 hours. The Hub stores one address per primitive, so when this
  session ends that address answers nothing until you re-run and paste a new one.
* **Each person who runs this gets a different URL.** One Hub, one address — so this is
  a demo you drive, not something visitors wire up themselves.

The next cell downloads `cloudflared` (Cloudflare's official release), because a Colab
VM has no inbound networking of its own.


In [ ]:
import importlib, subprocess, sys
from pathlib import Path

MODULE = "pvlib"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn[standard]", "pandas"], check=True)

# The archive is imported rather than installed, so nothing resolves its dependencies
# for us. They are declared in pyproject (pvlib has no requirements.txt, which is why
# the generic cell above found nothing to do), and reading them costs no build.
def _declared_deps(root):
    pyproject = Path(root) / "pyproject.toml"
    if pyproject.is_file():
        import tomllib

        table = tomllib.loads(pyproject.read_text())
        return list(table.get("project", {}).get("dependencies", []))
    requirements = Path(root) / "requirements.txt"
    if requirements.is_file():
        return [
            line.strip()
            for line in requirements.read_text().splitlines()
            if line.strip() and not line.startswith("#")
        ]
    return []


deps = _declared_deps(WORKDIR)
if deps:
    print(f"installing {len(deps)} declared dependencies …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)

# Import the archive off the path rather than pip-installing it. A published archive is
# a source tree with no .git, and a package whose version comes from setuptools_scm —
# pvlib does — fails at "Getting requirements to build wheel" for exactly that reason.
# Adding it to sys.path needs no build at all, and runs the published bytes rather than
# a rebuild of them.
sys.path.insert(0, WORKDIR)
for stale in [m for m in sys.modules if m == MODULE or m.startswith(MODULE + ".")]:
    del sys.modules[stale]

SERVING = ""
try:
    loaded = importlib.import_module(MODULE)
    # Colab preinstalls plenty; make sure this is the archive's copy and not one of
    # those, or "serving the published version" would be a lie.
    if WORKDIR in str(getattr(loaded, "__file__", "")):
        SERVING = "archive"
        print(f"serving the published version from {WORKDIR}")
    else:
        print(f"{MODULE} resolved to {loaded.__file__}, not the archive")
except Exception as exc:
    print("archive is not importable as-is:", exc)

if SERVING != "archive":
    print("falling back to PyPI pvlib — NOTE: no longer the published version")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pvlib"], check=True)
    SERVING = "pypi"

# Read from the checked-in service at generation time, so the notebook and the
# reference endpoint cannot drift apart.
Path("/content/endpoint.py").write_text('"""Reference Endpoint for the PowerAI Hub primitive @examples/pvlib-python.\n\nAnswers five of the tool interfaces that primitive declares, by calling pvlib for\nreal. Its only purpose is to show what an author hosts: the Hub publishes this\naddress and never calls it, so nothing in the catalog may depend on this process\nrunning.\n\n    uv run uvicorn main:app --port 8011\n\nUntrusted input crosses this boundary — anyone who can reach the URL can post to\nit — so every field is range-checked by pydantic before pvlib sees it, and the\nresponse carries no detail about this host.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom datetime import date as Date\nfrom typing import Annotated, Any, Callable, Literal\n\nimport pandas as pd\nfrom fastapi import FastAPI, Request\nfrom fastapi.responses import JSONResponse\nfrom pvlib import atmosphere, irradiance, solarposition\nfrom pvlib.location import Location\nfrom pydantic import BaseModel, Field, ValidationError\n\napp = FastAPI(title="pvlib reference endpoint", version="0.2.0")\n\n#: Protocol version, echoed on every reply. See docs/endpoint-protocol.md.\nPROTOCOL = "1"\n\n#: 15-minute steps: fine enough that the peak is not missed by much, coarse\n#: enough that a day is 96 rows. Each sample therefore covers a quarter hour,\n#: which is what turns a W/m² series into Wh/m².\nSAMPLE_FREQ = "15min"\nHOURS_PER_SAMPLE = 0.25\n\n\ndef _failure(code: str, message: str, status: int) -> JSONResponse:\n    """A failure the caller can act on, legible without the status code.\n\n    `ok` is the load-bearing field: Alfred hands the model the response *body as a\n    string*, so an agent never sees the HTTP status. Without `ok` a well-formed\n    error is indistinguishable from a result, and the agent reports a failure as an\n    answer.\n    """\n    return JSONResponse(\n        {"powerai": PROTOCOL, "ok": False, "error": {"code": code, "message": message}},\n        status_code=status,\n    )\n\n\n#: The tool named first here is also the legacy default (see ``_handle`` below) —\n#: it was this endpoint\'s only operation before the others existed, at a route\n#: (`/clearsky`) with no `tool` field to read.\nDISPATCH: dict[str, tuple[type[BaseModel], Callable[[Any], BaseModel]]] = {}\n\n\ndef _handle(body: Any) -> JSONResponse:\n    """Envelope handling shared by every declared tool, dispatched by ``tool``.\n\n    One route answers all of them — the same assumption the Hub\'s Action registry\n    and ``tool_endpoint`` already make (one address per primitive, not per tool) —\n    so the caller says which operation it wants via the envelope\'s `tool` field.\n\n    Answers either the enveloped or the bare form. Enveloped is the documented\n    protocol; bare arguments are what this endpoint accepted, as `/clearsky`, before\n    either the envelope or a second operation existed — so a bare request with no\n    `tool` field defaults to `clearsky_irradiance` rather than being rejected,\n    keeping anything already built against the old shape working.\n\n    A `ValueError` from a compute function is reported as the caller\'s mistake\n    (``invalid_input``) rather than the server\'s (``internal``) — it is how a\n    compute function signals "your arguments parsed, but do not mean anything\n    physical" (an unparseable timestamp, a zenith angle at the horizon), which\n    pydantic\'s field-level checks cannot express.\n    """\n    if not isinstance(body, dict):\n        return _failure("invalid_input", "Request body must be a JSON object.", 400)\n\n    enveloped = bool(body.get("powerai"))\n    tool = body.get("tool") if enveloped else next(iter(DISPATCH))\n    if tool not in DISPATCH:\n        known = ", ".join(sorted(DISPATCH))\n        return _failure("unknown_tool", f"`tool` must be one of: {known}.", 400)\n    request_model, compute = DISPATCH[tool]\n\n    arguments = body.get("input") if enveloped else body\n    if not isinstance(arguments, dict):\n        return _failure("invalid_input", "`input` must be a JSON object.", 400)\n\n    try:\n        parsed = request_model.model_validate(arguments)\n    except ValidationError as exc:\n        first = exc.errors()[0]\n        field = ".".join(str(p) for p in first.get("loc", ())) or "input"\n        return _failure("invalid_input", f"{field}: {first.get(\'msg\', \'invalid\')}", 400)\n\n    try:\n        result = compute(parsed)\n    except ValueError as exc:\n        return _failure("invalid_input", str(exc), 400)\n    except Exception:  # noqa: BLE001 — never leak a stack trace or host detail\n        return _failure("internal", "The calculation failed.", 500)\n\n    return JSONResponse({"powerai": PROTOCOL, "ok": True, "output": result.model_dump()})\n\n\n# --- clearsky_irradiance ------------------------------------------------------\n\n\nclass ClearskyRequest(BaseModel):\n    latitude: Annotated[float, Field(ge=-90, le=90, description="Degrees north")]\n    longitude: Annotated[float, Field(ge=-180, le=180, description="Degrees east")]\n    date: Annotated[Date, Field(description="UTC day, YYYY-MM-DD")]\n    # pvlib also offers \'haurwitz\', but it returns GHI *only* — no DNI or DHI. It is\n    # excluded rather than filled with nulls, because the tool interface this\n    # endpoint answers declares all three, and a contract that is true for some\n    # inputs is not true.\n    model: Literal["ineichen", "simplified_solis"] = "ineichen"\n\n\nclass ClearskyResponse(BaseModel):\n    ghi_peak_w_m2: float\n    dni_peak_w_m2: float\n    dhi_peak_w_m2: float\n    ghi_daily_wh_m2: float\n    model: str\n\n\ndef clearsky(req: ClearskyRequest) -> ClearskyResponse:\n    """Peak and daily-total clear-sky irradiance for one UTC day at one place."""\n    location = Location(req.latitude, req.longitude, tz="UTC")\n    times = pd.date_range(\n        start=f"{req.date} 00:00", end=f"{req.date} 23:59", freq=SAMPLE_FREQ, tz="UTC"\n    )\n    frame = location.get_clearsky(times, model=req.model)\n    return ClearskyResponse(\n        ghi_peak_w_m2=round(float(frame["ghi"].max()), 1),\n        dni_peak_w_m2=round(float(frame["dni"].max()), 1),\n        dhi_peak_w_m2=round(float(frame["dhi"].max()), 1),\n        ghi_daily_wh_m2=round(float(frame["ghi"].sum()) * HOURS_PER_SAMPLE, 1),\n        model=req.model,\n    )\n\n\n# --- solar_position ------------------------------------------------------------\n\n\nclass SolarPositionRequest(BaseModel):\n    latitude: Annotated[float, Field(ge=-90, le=90, description="Degrees north")]\n    longitude: Annotated[float, Field(ge=-180, le=180, description="Degrees east")]\n    time: Annotated[\n        str, Field(description="UTC ISO-8601 timestamp, e.g. 2026-06-21T18:00:00")\n    ]\n\n\nclass SolarPositionResponse(BaseModel):\n    zenith_deg: float\n    azimuth_deg: float\n    elevation_deg: float\n\n\ndef solar_position(req: SolarPositionRequest) -> SolarPositionResponse:\n    """Apparent sun position for one UTC instant at one place."""\n    try:\n        times = pd.DatetimeIndex([req.time])\n    except Exception as exc:\n        raise ValueError(f"time: could not parse \'{req.time}\' as a timestamp") from exc\n    if times.tz is None:\n        times = times.tz_localize("UTC")\n    frame = solarposition.get_solarposition(times, req.latitude, req.longitude)\n    row = frame.iloc[0]\n    return SolarPositionResponse(\n        zenith_deg=round(float(row["apparent_zenith"]), 2),\n        azimuth_deg=round(float(row["azimuth"]), 2),\n        elevation_deg=round(float(row["apparent_elevation"]), 2),\n    )\n\n\n# --- extraterrestrial_irradiance ------------------------------------------------\n\n\nclass ExtraRadiationRequest(BaseModel):\n    date: Annotated[Date, Field(description="UTC day, YYYY-MM-DD")]\n\n\nclass ExtraRadiationResponse(BaseModel):\n    extraterrestrial_irradiance_w_m2: float\n\n\ndef extraterrestrial_irradiance(req: ExtraRadiationRequest) -> ExtraRadiationResponse:\n    """Top-of-atmosphere solar irradiance for one UTC day.\n\n    Varies through the year only because Earth-Sun distance does — this is the\n    same input every clear-sky and transposition model scales from, not a\n    location-dependent quantity, which is why it takes no latitude or longitude.\n    """\n    value = irradiance.get_extra_radiation(pd.Timestamp(req.date))\n    return ExtraRadiationResponse(extraterrestrial_irradiance_w_m2=round(float(value), 1))\n\n\n# --- relative_airmass ------------------------------------------------------------\n\n\nclass AirmassRequest(BaseModel):\n    zenith_deg: Annotated[float, Field(ge=0, le=90, description="Solar zenith angle, degrees")]\n\n\nclass AirmassResponse(BaseModel):\n    relative_airmass: float\n\n\ndef relative_airmass(req: AirmassRequest) -> AirmassResponse:\n    """Relative (sea-level) airmass for a given solar zenith angle.\n\n    Undefined once the sun is at the horizon, so a non-finite result — reachable\n    even inside the field\'s declared range, right at its edge — is rejected rather\n    than returned: a contract that is true for some inputs is not true.\n    """\n    value = atmosphere.get_relative_airmass(req.zenith_deg)\n    if not math.isfinite(value):\n        raise ValueError("zenith_deg: airmass is undefined this close to the horizon")\n    return AirmassResponse(relative_airmass=round(float(value), 3))\n\n\n# --- plane_of_array_irradiance --------------------------------------------------\n\n\nclass PoaRequest(BaseModel):\n    surface_tilt_deg: Annotated[\n        float, Field(ge=0, le=180, description="Panel tilt from horizontal, degrees")\n    ]\n    surface_azimuth_deg: Annotated[\n        float, Field(ge=0, le=360, description="Panel azimuth, clockwise from north, degrees")\n    ]\n    solar_zenith_deg: Annotated[float, Field(ge=0, le=180, description="Solar zenith angle")]\n    solar_azimuth_deg: Annotated[\n        float, Field(ge=0, le=360, description="Solar azimuth, clockwise from north, degrees")\n    ]\n    dni_w_m2: Annotated[float, Field(ge=0, description="Direct normal irradiance")]\n    ghi_w_m2: Annotated[float, Field(ge=0, description="Global horizontal irradiance")]\n    dhi_w_m2: Annotated[float, Field(ge=0, description="Diffuse horizontal irradiance")]\n\n\nclass PoaResponse(BaseModel):\n    poa_global_w_m2: float\n    poa_direct_w_m2: float\n    poa_diffuse_w_m2: float\n\n\ndef plane_of_array_irradiance(req: PoaRequest) -> PoaResponse:\n    """Irradiance on a tilted panel, from known sun position and horizontal irradiance.\n\n    Uses pvlib\'s isotropic sky-diffuse model — the simplest of several, and the one\n    that needs no parameters beyond what this schema already declares.\n    """\n    result = irradiance.get_total_irradiance(\n        surface_tilt=req.surface_tilt_deg,\n        surface_azimuth=req.surface_azimuth_deg,\n        solar_zenith=req.solar_zenith_deg,\n        solar_azimuth=req.solar_azimuth_deg,\n        dni=req.dni_w_m2,\n        ghi=req.ghi_w_m2,\n        dhi=req.dhi_w_m2,\n    )\n    return PoaResponse(\n        poa_global_w_m2=round(float(result["poa_global"]), 1),\n        poa_direct_w_m2=round(float(result["poa_direct"]), 1),\n        poa_diffuse_w_m2=round(float(result["poa_diffuse"]), 1),\n    )\n\n\n# --- the one route ----------------------------------------------------------\n#\n# clearsky_irradiance is listed first: it is also the legacy default `_handle` falls\n# back to for a bare (pre-envelope, pre-multi-tool) request with no `tool` field.\n\nDISPATCH.update(\n    {\n        "clearsky_irradiance": (ClearskyRequest, clearsky),\n        "solar_position": (SolarPositionRequest, solar_position),\n        "extraterrestrial_irradiance": (ExtraRadiationRequest, extraterrestrial_irradiance),\n        "relative_airmass": (AirmassRequest, relative_airmass),\n        "plane_of_array_irradiance": (PoaRequest, plane_of_array_irradiance),\n    }\n)\n\n\n@app.post("/run")\nasync def run(request: Request) -> JSONResponse:\n    try:\n        body = await request.json()\n    except Exception:  # noqa: BLE001 — malformed JSON is the caller\'s mistake, not a crash\n        return _failure("invalid_input", "Request body is not valid JSON.", 400)\n    return _handle(body)\n\n\n@app.get("/health")\ndef health() -> dict[str, str]:\n    return {"status": "ok"}\n\n\ndef _self_check() -> None:\n    """Assert the physics is the right order of magnitude, not just that it runs.\n\n    A bad unit conversion or a mixed-up column still returns a plausible-looking\n    float, so the checks below pin behaviour that only holds if the calculation is\n    actually right.\n    """\n    # clearsky_irradiance: summer beats winter, the equator beats the Arctic in\n    # December, and the daily total is consistent with the peak.\n    summer = clearsky(ClearskyRequest(latitude=37.0, longitude=-122.0, date=Date(2026, 6, 21)))\n    winter = clearsky(ClearskyRequest(latitude=37.0, longitude=-122.0, date=Date(2026, 12, 21)))\n    assert 700 < summer.ghi_peak_w_m2 < 1200, summer\n    assert summer.ghi_peak_w_m2 > winter.ghi_peak_w_m2, (summer, winter)\n    assert summer.ghi_daily_wh_m2 > winter.ghi_daily_wh_m2, (summer, winter)\n    assert summer.ghi_peak_w_m2 < summer.ghi_daily_wh_m2 < summer.ghi_peak_w_m2 * 24, summer\n    polar = clearsky(ClearskyRequest(latitude=78.0, longitude=15.0, date=Date(2026, 12, 21)))\n    assert polar.ghi_peak_w_m2 == 0.0, polar\n    solis = clearsky(\n        ClearskyRequest(\n            latitude=37.0, longitude=-122.0, date=Date(2026, 6, 21), model="simplified_solis"\n        )\n    )\n    assert solis.ghi_peak_w_m2 != summer.ghi_peak_w_m2, (solis, summer)\n    assert solis.dni_peak_w_m2 > 0 and solis.dhi_peak_w_m2 > 0, solis\n    print(f"clearsky        summer={summer.ghi_peak_w_m2} winter={winter.ghi_peak_w_m2}")\n\n    # solar_position: local solar noon beats deep night, at the same place.\n    noon = solar_position(\n        SolarPositionRequest(latitude=37.0, longitude=-122.0, time="2026-06-21T20:00:00")\n    )\n    night = solar_position(\n        SolarPositionRequest(latitude=37.0, longitude=-122.0, time="2026-06-21T08:00:00")\n    )\n    assert 0 <= noon.zenith_deg < 90, noon  # sun above the horizon near local noon\n    assert noon.zenith_deg < night.zenith_deg, (noon, night)  # higher at noon than at night\n    print(f"solar_position  noon_zenith={noon.zenith_deg} night_zenith={night.zenith_deg}")\n\n    # extraterrestrial_irradiance: within the sun\'s real ~1321-1414 W/m^2 range at\n    # Earth\'s distance, and January (near perihelion) beats July (near aphelion).\n    january = extraterrestrial_irradiance(ExtraRadiationRequest(date=Date(2026, 1, 4)))\n    july = extraterrestrial_irradiance(ExtraRadiationRequest(date=Date(2026, 7, 4)))\n    assert 1300 < january.extraterrestrial_irradiance_w_m2 < 1420, january\n    assert january.extraterrestrial_irradiance_w_m2 > july.extraterrestrial_irradiance_w_m2, (\n        january,\n        july,\n    )\n    print(\n        "extraterrestrial_irradiance "\n        f"jan={january.extraterrestrial_irradiance_w_m2} jul={july.extraterrestrial_irradiance_w_m2}"\n    )\n\n    # relative_airmass: exactly 1 straight overhead, larger at a slant, undefined\n    # at the horizon.\n    overhead = relative_airmass(AirmassRequest(zenith_deg=0.0))\n    slant = relative_airmass(AirmassRequest(zenith_deg=60.0))\n    assert math.isclose(overhead.relative_airmass, 1.0, abs_tol=1e-6), overhead\n    assert slant.relative_airmass > overhead.relative_airmass, (overhead, slant)\n    try:\n        relative_airmass(AirmassRequest(zenith_deg=90.0))\n        raise AssertionError("airmass at the horizon should be rejected, not returned")\n    except ValueError:\n        pass\n    print(f"relative_airmass overhead={overhead.relative_airmass} slant={slant.relative_airmass}")\n\n    # plane_of_array_irradiance: a flat panel matches horizontal irradiance exactly\n    # (its plane *is* horizontal), and tilting toward the sun gains over flat.\n    flat = plane_of_array_irradiance(\n        PoaRequest(\n            surface_tilt_deg=0,\n            surface_azimuth_deg=180,\n            solar_zenith_deg=30,\n            solar_azimuth_deg=180,\n            dni_w_m2=summer.dni_peak_w_m2,\n            ghi_w_m2=summer.ghi_peak_w_m2,\n            dhi_w_m2=summer.dhi_peak_w_m2,\n        )\n    )\n    assert math.isclose(flat.poa_global_w_m2, summer.ghi_peak_w_m2, abs_tol=0.1), flat\n    tilted_at_sun = plane_of_array_irradiance(\n        PoaRequest(\n            surface_tilt_deg=30,\n            surface_azimuth_deg=180,\n            solar_zenith_deg=30,\n            solar_azimuth_deg=180,\n            dni_w_m2=summer.dni_peak_w_m2,\n            ghi_w_m2=summer.ghi_peak_w_m2,\n            dhi_w_m2=summer.dhi_peak_w_m2,\n        )\n    )\n    assert tilted_at_sun.poa_global_w_m2 > flat.poa_global_w_m2, (flat, tilted_at_sun)\n    print(f"poa_irradiance  flat={flat.poa_global_w_m2} tilted_at_sun={tilted_at_sun.poa_global_w_m2}")\n\n    print("self-check OK")\n\n\nif __name__ == "__main__":\n    _self_check()\n')
print("wrapper written to /content/endpoint.py")


In [ ]:
import os, re, subprocess, sys, time, urllib.request

PORT = 8011
ROUTE = "/run"  # every declared tool answers here; the request's own
                            # `tool` field says which one — see the check cell below.
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Re-running this cell has to be safe, and without this it is not. A previous run
# leaves a tunnel and a server alive, so the next one would: fail to overwrite the
# cloudflared binary while it is executing ("Text file busy"), fail to bind the port,
# and then pass the health probe anyway — because the *old* server answered it. That
# last part is the dangerous one: a stale process serving a stale archive, reported as
# a fresh success.
for pattern in ("cloudflared tunnel", "uvicorn endpoint:app"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    # Colab VMs have no inbound networking, so a tunnel is the only way in. Quiet
    # unless it fails, and then loud: `check=True` alone raises with no detail.
    got = subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, capture_output=True, text=True,
    )
    if got.returncode != 0:
        raise RuntimeError(f"could not install cloudflared:\n{got.stderr[-1000:]}")
    print("cloudflared installed")
else:
    print("cloudflared already present")

# sys.path does not cross into a child process, so the archive has to be handed over
# explicitly — otherwise this server imports whatever is installed globally while the
# cell above reports it is serving the published version.
env = dict(os.environ)
if SERVING == "archive":
    env["PYTHONPATH"] = WORKDIR + os.pathsep + env.get("PYTHONPATH", "")

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "endpoint:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="/content", env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait for the app before exposing it, so the tunnel never fronts a dead port.
for _ in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("the endpoint did not come up; check the cell above")
print(f"endpoint up on :{PORT}")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = ""
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; re-run this cell")

ENDPOINT_URL = PUBLIC_URL + ROUTE
print()
print("Endpoint URL (paste into the Tool tab -> Configure endpoint):")
print("   ", ENDPOINT_URL)


In [ ]:
import json, urllib.error, urllib.request

# Built from the tool interface the Hub publishes for @examples/pvlib-python; edit `arguments` freely.
arguments = {"latitude": 37.0, "longitude": -122.0, "date": "2026-06-21"}
envelope = {"powerai": "1", "tool": "clearsky_irradiance", "input": arguments}

request = urllib.request.Request(
    ENDPOINT_URL, data=json.dumps(envelope).encode(),
    headers={"Content-Type": "application/json"},
)
reply = json.load(urllib.request.urlopen(request, timeout=90))
print(json.dumps(reply, indent=1))
assert reply.get("ok") is True, reply

# And that a bad request comes back as ok:false rather than as something a caller
# could mistake for a result — the field an agent platform actually reads.
print()
print("Sending a deliberately broken request now — a rejection below is the pass:")
broken = dict(envelope, input={k: v for k, v in arguments.items() if k != "latitude"})
try:
    urllib.request.urlopen(urllib.request.Request(
        ENDPOINT_URL, data=json.dumps(broken).encode(),
        headers={"Content-Type": "application/json"}), timeout=30)
    raise AssertionError("a request missing a required field should not succeed")
except urllib.error.HTTPError as err:
    failure = json.load(err)
    assert failure["ok"] is False, failure
    print("Correctly rejected ->", failure["error"]["message"])

# This endpoint answers more than one operation at the same URL — the request's own
# `tool` field says which. Valid arguments for "clearsky_irradiance" were hand-verified above;
# the generator does not know the rest, so this checks something weaker but still
# real: each is recognised and validates its own input, rather than silently falling
# through to "clearsky_irradiance" or a generic 404.
print()
for name in ['solar_position', 'extraterrestrial_irradiance', 'relative_airmass', 'plane_of_array_irradiance']:
    probe = urllib.request.Request(
        ENDPOINT_URL, data=json.dumps({"powerai": "1", "tool": name, "input": {}}).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        urllib.request.urlopen(probe, timeout=30)
        raise AssertionError(f"{name}: empty input should have been rejected, not accepted")
    except urllib.error.HTTPError as err:
        result = json.load(err)
        assert result.get("ok") is False, result
        assert result["error"].get("code") != "unknown_tool", (
            f"{name} was not recognised — check the spelling against its "
            "tool_interface.name"
        )
        print(f"{name} -> recognised, validates its own input ({result['error']['message']})")

print()
print("endpoint verified")


### Register it, then clean up

Paste the URL above into the [Tool tab](https://hub.powerai.ai/examples/pvlib-python?tab=tool) → *Configure endpoint*. The
primitive then reports `invocable: true`, appears in `?invocable=true`, and is rendered
as an action in `/api/integrations/action-registry/?as=markdown` for an agent platform
to pull.

**When you finish, clear it** — the catalogue should not advertise an address that died
with this session. Same dialog: *Change endpoint* → **Remove**.

Clear it there rather than with `curl`. That API is owner-authenticated, so a bare
request is rejected, and the credential that would satisfy it does not belong in a
notebook this repo publishes.


## Your turn

`@examples/pvlib-python` is unpacked in `/content/primitive` and it is the working directory. Add a
cell and use it.

Nothing here is sent back to the Hub — this session is yours, and it disappears when
you close it. Colab's free tier gives you CPU always and a GPU when one is spare.
